In [4]:
import pandas as pd
from pathlib import Path

MODELS = {
    'demo_only':      Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demographics_seaad"),
    'genes_only':     Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"),
    'genes_apoe':     Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_seaad"),
    'genes_demo':     Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_demographics_seaad"),
}
CELL_TYPES = ['Ast', 'Mic', 'Inh', 'Oli', 'Ex', 'Opc']

rows = []
for model_name, base in MODELS.items():
    for ct in CELL_TYPES:
        for split in range(1, 6):
            p = base / ct / f"split_{split}" / "output_csv.csv"
            if not p.exists():
                continue
            df = pd.read_csv(p)
            df['model'] = model_name
            df['cell_type'] = ct
            df['split'] = split
            rows.append(df)

all_results = pd.concat(rows, ignore_index=True)

# Mean test AUC per (model, cell_type) across splits
summary = (all_results.groupby(['cell_type', 'model'])
           .agg(test_auc_mean=('test_roc_auc', 'mean'),
                test_auc_std=('test_roc_auc', 'std'),
                n_splits=('test_roc_auc', 'size'))
           .reset_index())
print(summary.pivot(index='cell_type', columns='model', values='test_auc_mean').round(3))

model      demo_only  genes_apoe  genes_demo  genes_only
cell_type                                               
Ast            0.500       0.521       0.622       0.508
Ex             0.511       0.536       0.618       0.549
Inh            0.560       0.543       0.624       0.507
Mic            0.493       0.607       0.635       0.566
Oli            0.548       0.509       0.598       0.573
Opc            0.678       0.676       0.704       0.589


In [5]:
import joblib
from pathlib import Path
from collections import defaultdict

FULL_BASE = Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_demographics_seaad")
CELL_TYPES = ['Ast', 'Mic', 'Inh', 'Oli', 'Ex', 'Opc']

for cell_type in CELL_TYPES:
    gene_presence = defaultdict(int)
    for split in range(1, 6):
        path = FULL_BASE / cell_type / f"split_{split}" / "maximal_classifier.joblib"
        if not path.exists():
            continue
        model = joblib.load(path)
        for feat, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence[feat] += 1

    predictors = sorted([g for g, c in gene_presence.items() if c >= 2])
    print(f"\n{'='*60}")
    print(f"{cell_type}: {len(predictors)} predictors (n_splits >= 2)")
    print(f"{'='*60}")
    print(predictors)


Ast: 28 predictors (n_splits >= 2)
[np.str_('AC0089572'), np.str_('AC0124041'), np.str_('APOEGenotype_23'), np.str_('APOEGenotype_24'), np.str_('APOEGenotype_33'), np.str_('APOEGenotype_34'), np.str_('APOEGenotype_44'), np.str_('ARL17B'), np.str_('AgeatDeath'), np.str_('C1orf61'), np.str_('CCBE1'), np.str_('FAM107A'), np.str_('FTH1'), np.str_('HSPA1B'), np.str_('MACF1'), np.str_('MALAT1'), np.str_('MTATP6'), np.str_('MTCO2'), np.str_('MTCO3'), np.str_('MTCYB'), np.str_('NEAT1'), np.str_('PFKP'), np.str_('PLCG2'), np.str_('RANBP3L'), np.str_('SLC39A11'), np.str_('Sex'), np.str_('UTY'), np.str_('XIST')]

Mic: 134 predictors (n_splits >= 2)
[np.str_('AC0072622'), np.str_('AC0086911'), np.str_('AC0112871'), np.str_('AC0234691'), np.str_('AC0938494'), np.str_('AL0354462'), np.str_('ALCAM'), np.str_('ALOX5AP'), np.str_('ANKRD22'), np.str_('AP0016363'), np.str_('AP0030861'), np.str_('APOC1'), np.str_('APOEGenotype_22'), np.str_('APOEGenotype_23'), np.str_('APOEGenotype_24'), np.str_('APOEGen

In [6]:
import joblib
from pathlib import Path
from collections import defaultdict

FULL_BASE = Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_demographics_seaad")
CELL_TYPES = ['Ast', 'Mic', 'Inh', 'Oli', 'Ex', 'Opc']

# non-gene features in this model
COVARIATES = {'Sex', 'AgeatDeath'} | {f'APOEGenotype_{g}' for g in ['22','23','24','33','34','44']}

for cell_type in CELL_TYPES:
    gene_presence = defaultdict(int)
    for split in range(1, 6):
        path = FULL_BASE / cell_type / f"split_{split}" / "maximal_classifier.joblib"
        if not path.exists():
            continue
        model = joblib.load(path)
        for feat, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0 and feat not in COVARIATES:
                gene_presence[feat] += 1

    predictors = sorted([g for g, c in gene_presence.items() if c >= 2])
    print(f"\n{'='*60}")
    print(f"{cell_type}: {len(predictors)} gene predictors (n_splits >= 2)")
    print(f"{'='*60}")
    print(predictors)


Ast: 21 gene predictors (n_splits >= 2)
[np.str_('AC0089572'), np.str_('AC0124041'), np.str_('ARL17B'), np.str_('C1orf61'), np.str_('CCBE1'), np.str_('FAM107A'), np.str_('FTH1'), np.str_('HSPA1B'), np.str_('MACF1'), np.str_('MALAT1'), np.str_('MTATP6'), np.str_('MTCO2'), np.str_('MTCO3'), np.str_('MTCYB'), np.str_('NEAT1'), np.str_('PFKP'), np.str_('PLCG2'), np.str_('RANBP3L'), np.str_('SLC39A11'), np.str_('UTY'), np.str_('XIST')]

Mic: 126 gene predictors (n_splits >= 2)
[np.str_('AC0072622'), np.str_('AC0086911'), np.str_('AC0112871'), np.str_('AC0234691'), np.str_('AC0938494'), np.str_('AL0354462'), np.str_('ALCAM'), np.str_('ALOX5AP'), np.str_('ANKRD22'), np.str_('AP0016363'), np.str_('AP0030861'), np.str_('APOC1'), np.str_('ARHGAP24'), np.str_('ARHGAP26'), np.str_('ARL17B'), np.str_('ATP8B4'), np.str_('C1QB'), np.str_('C1QC'), np.str_('C3'), np.str_('CACNA1A'), np.str_('CCDC26'), np.str_('CD14'), np.str_('CLECL1'), np.str_('CPA6'), np.str_('CPVL'), np.str_('CTSB'), np.str_('DLEU1